#Text Generation

In [ ]:
!pip install azure-core azure-ai-formrecognizer

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.ai.formrecognizer import AnalysisFeature
from collections import Counter

endpoint = "YOUR_API_ENDPOINT_HERE"
key = "YOUR_API_KEY_HERE"

local_file_path = "/content/Raj Kumar.png"
document_analysis_client = DocumentAnalysisClient(
    endpoint=endpoint, credential=AzureKeyCredential(key)
)

with open(local_file_path, "rb") as file:
    poller = document_analysis_client.begin_analyze_document("prebuilt-document", document=file, features=[
                AnalysisFeature.STYLE_FONT
            ])
    result = poller.result()

fonts = []

for style in result.styles:
    if style.similar_font_family:
        font = style.similar_font_family.split(",")[0].strip()
        fonts.append(font)

font_counts = Counter(fonts)
majority_font = font_counts.most_common(1)[0][0]

print("Majority Font:", majority_font)

bounding_box_values = {}
for kv_pair in result.key_value_pairs:
    if kv_pair.key:
        key_content = kv_pair.key.content
        key_bounding_box = kv_pair.key.bounding_regions[0].polygon if kv_pair.key.bounding_regions else None
        print(f"Key: '{key_content}'")
        print(f"  Key Bounding Box: {key_bounding_box}")

    if kv_pair.value:
        value_content = kv_pair.value.content
        value_bounding_box = kv_pair.value.bounding_regions[0].polygon if kv_pair.value.bounding_regions else None
        print(f"Value: '{value_content}'")
        print(f"  Value Bounding Box: {value_bounding_box}")
        bounding_box_values[value_content] = value_bounding_box

In [ ]:
bounding_box_values

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

image = cv2.imread(local_file_path)

for kv_pair in result.key_value_pairs:
    if kv_pair.value and kv_pair.value.bounding_regions:
        bounding_box = kv_pair.value.bounding_regions[0].polygon
        if bounding_box:
            points = np.array([[int(point.x), int(point.y)] for point in bounding_box], np.int32)
            points = points.reshape((-1, 1, 2))

            cv2.polylines(image, [points], isClosed=True, color=(255, 255, 0), thickness=2)

    if kv_pair.key and kv_pair.key.bounding_regions:
        bounding_box = kv_pair.key.bounding_regions[0].polygon
        if bounding_box:
            points = np.array([[int(point.x), int(point.y)] for point in bounding_box], np.int32)
            points = points.reshape((-1, 1, 2))

            cv2.polylines(image, [points], isClosed=True, color=(255, 255, 0), thickness=2)


output_file_path = "/content/Image_with_bounding_boxes.png"
cv2.imwrite(output_file_path, image)
cv2_imshow(image)

In [ ]:
result.key_value_pairs

In [ ]:
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

client = OpenAI(
    base_url="https://api.chatanywhere.tech/v1"
)

messages = [
    {"role": "system", "content": "You are a helpful AI assistant."}
]

user_input = f"""
    I performed OCR recognition on an identity document and extracted the following text:
    {result.key_value_pairs}
    For each field in the text, indicate whether it is static (unchangeable) or dynamic (changeable) in the context of generating multiple copies for different users.
    Please follow this format for the output:
    <output><text>Field Name:<s>Bounding Box of Value from key_value_pairs (only provide the list of coordinates)</s><s>YES</s></text> (for static fields)
    <output><text>Field Name:<s>Bounding Box of Value from key_value_pairs (only provide the list of coordinates)</s><s>NO</s></text> (for dynamic fields)
    Important:
    1. Give the bounding box structure as it is given in the key_value_pairs, starting from polygon.
    2. Please give it in the above format only, don't change anything, even <s> and </s>. If you find anything that is not valid, please remove it from the final output. You are very intelligent; only consider information pairs that are dynamic (i.e., can change from one document to another) and are related to the user only.
    3. Don't leave out any valid key-value pair that is related to the user.
    """

messages.append({"role": "user", "content": user_input})

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    temperature=0.7
)

reply = response.choices[0].message.content
print(reply)

In [ ]:
import re
pattern = r"<text>(.*?)<s>(.*?)</s><s>(.*?)</s></text>"
matches = re.findall(pattern, reply)

In [ ]:
matches

In [ ]:
image = cv2.imread(local_file_path)
for tup in matches:
        bounding_box_str = tup[1]
        if bounding_box_str:
            points_str = re.findall(r"Point\(x=(\d+\.\d+), y=(\d+\.\d+)\)", bounding_box_str)
            points = np.array([[int(float(x)), int(float(y))] for x, y in points_str], np.int32)
            points = points.reshape((-1, 1, 2))
            cv2.polylines(image, [points], isClosed=True, color=(255, 0, 255), thickness=4)

output_file_path = "/content/dynamic_data_bounding_boxes.png"
cv2.imwrite(output_file_path, image)
cv2_imshow(image)

In [ ]:
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

client = OpenAI(
    base_url="https://api.chatanywhere.tech/v1"
)

messages = [
    {"role": "system", "content": "You are a helpful AI assistant."}
]

user_input = f"""
    I performed OCR recognition on an identity document (PASSPORT) and extracted the following text:
    {matches}
    For each field in the text, generate a realistic dummy value corresponding to that field and return the output in the following format:
    <output><text>here data you generated:<s>here bounding box coordinates</s></text></output>
    Important:
    1. Only provide the generated data values; do not include the field names.
    2. If the field name has two or more inputs separated by '/' then provide the dummy values separated by '/'. (Example- Date of Birth/Time: (DD/MM/YYYY)/(Time with AM or PM)).
    3. Follow the exact format above and do not change anything, including <s> and </s>.
    4. Generate different dummy data each time (avoid repeating similar values across iterations).
    5. The generated data must be in English.
    6. Process all elements in the text provided.
    """

messages.append({"role": "user", "content": user_input})

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    temperature=0.7
)

dummy = response.choices[0].message.content
print(dummy)

In [ ]:
pattern_values = r"<text>(.*?)<s>(.*?)</s></text>"
dummy_data = re.findall(pattern_values, dummy)

In [ ]:
dummy_data

In [ ]:
from PIL import Image, ImageDraw, ImageFont

image_path = "/content/Raj Kumar.png"
image_pil = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image_pil)

def parse_points(point_string):
    clean_string = re.sub(r'Point\(x=([\d.]+), y=([\d.]+)\)', r'(\1, \2)', point_string)
    points = eval(clean_string)
    return [(int(float(x)), int(float(y))) for x, y in points]

font_path = "/content/Courier New.ttf"
font = ImageFont.truetype(font_path, 28)

for text, points_string in dummy_data:
    points = parse_points(points_string)
    x1, y1 = points[0]
    x2, y2 = points[2]
    draw.rectangle([x1, y1, x2, y2], fill="white")
    draw.text((x1, y1), text, fill="black", font=font, stroke_width=0.4)

output_path = "synthetic_image.png"
image_pil.save(output_path)
cv2_imshow(np.array(image_pil))